[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aurelio-labs/semantic-router/blob/main/docs/01-save-load-from-file.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/aurelio-labs/semantic-router/blob/main/docs/01-save-load-from-file.ipynb)

# Routers from File

Here we will show how to save a router to a YAML or JSON file, and how to load it back.

## Getting Started

We start by installing the library:

In [ ]:
!pip install -qU "semantic-router[fastembed]>=1.0.0"

## Define Routes

First let's create a list of routes:

In [1]:
from semantic_router import Route

politics = Route(
    name="politics",
    utterances=[
        "isn't politics the best thing ever",
        "why don't you tell me about your political opinions",
        "don't you just love the president",
        "don't you just hate the president",
        "they're going to destroy this country!",
        "they will save the country!",
    ],
)
chitchat = Route(
    name="chitchat",
    utterances=[
        "how's the weather today?",
        "how are things going?",
        "lovely weather today",
        "the weather is horrendous",
        "let's go to the chippy",
    ],
)

routes = [politics, chitchat]

We build a router with these routes, a local encoder, and a confidence threshold.

In [2]:
from semantic_router import IndexedRouter
from semantic_router.encoders import FastEmbedEncoder

encoder = FastEmbedEncoder(name="BAAI/bge-small-en-v1.5")

router = IndexedRouter(encoder=encoder, routes=routes, confidence_threshold=0.6)

## Test Route

In [3]:
router("isn't politics the best thing ever")

RouteChoice(name='politics', confidence=0.7211165634578482, calibrated=False)

In [4]:
router("how's the weather today?")

RouteChoice(name='chitchat', confidence=0.8048876076635829, calibrated=False)

## Save to YAML

To save our router we call the `to_yaml` method:

In [5]:
router.to_yaml(path="router.yaml")

We can view the file we just saved to see what information is stored:

In [6]:
with open("router.yaml") as f:
    print(f.read())

router_type: indexed
encoder:
  type: fastembed
  name: BAAI/bge-small-en-v1.5
sparse_encoder: null
alpha: 0.3
top_k: 5
aggregation: mean
index:
  type: local
confidence_threshold: 0.6
calibrator: null
routes:
- name: politics
  utterances:
  - isn't politics the best thing ever
  - why don't you tell me about your political opinions
  - don't you just love the president
  - don't you just hate the president
  - they're going to destroy this country!
  - they will save the country!
  description: null
  confidence_threshold: null
  metadata: {}
- name: chitchat
  utterances:
  - how's the weather today?
  - how are things going?
  - lovely weather today
  - the weather is horrendous
  - let's go to the chippy
  description: null
  confidence_threshold: null
  metadata: {}



The `router_type` tells the loader which router to build. The rest is everything needed to rebuild it: the encoder, the index, the threshold and the routes. API keys are never written to the file; they come from the environment when the router is loaded.

## Load from YAML

To load it, we use `load_router`, which returns whichever router the file describes:

In [7]:
from semantic_router import load_router

router = load_router(path="router.yaml")
type(router).__name__

'IndexedRouter'

Loading re-embeds the routes into a fresh index, so the router is ready to use straight away:

In [8]:
for query in [
    "isn't politics the best thing ever",
    "how's the weather today?",
    "I'm interested in learning about llama 2",
]:
    print(f"{query} -> {router(query).name}")

isn't politics the best thing ever -> politics
how's the weather today? -> chitchat
I'm interested in learning about llama 2 -> None


## JSON

JSON works the same way, with `to_json`. `load_router` picks the format from the file extension.

In [9]:
import json

router.to_json(path="router.json")

with open("router.json") as f:
    print(json.dumps(json.load(f), indent=2))

{
  "router_type": "indexed",
  "encoder": {
    "type": "fastembed",
    "name": "BAAI/bge-small-en-v1.5"
  },
  "sparse_encoder": null,
  "alpha": 0.3,
  "top_k": 5,
  "aggregation": "mean",
  "index": {
    "type": "local"
  },
  "confidence_threshold": 0.6,
  "calibrator": null,
  "routes": [
    {
      "name": "politics",
      "utterances": [
        "isn't politics the best thing ever",
        "why don't you tell me about your political opinions",
        "don't you just love the president",
        "don't you just hate the president",
        "they're going to destroy this country!",
        "they will save the country!"
      ],
      "description": null,
      "confidence_threshold": null,
      "metadata": {}
    },
    {
      "name": "chitchat",
      "utterances": [
        "how's the weather today?",
        "how are things going?",
        "lovely weather today",
        "the weather is horrendous",
        "let's go to the chippy"
      ],
      "description": null,


In [10]:
router = load_router(path="router.json")
router("how's the weather today?")

RouteChoice(name='chitchat', confidence=0.8048876076635829, calibrated=False)

## Saving a Calibrated Router

A router calibrated with `fit()` saves its calibrator too, so a loaded router is calibrated without fitting again. We fit on a few held-out labelled queries (see [route calibration](https://github.com/aurelio-labs/semantic-router/blob/main/docs/06-calibration.ipynb)):

In [11]:
X = [
    "What's your opinion on the current government?",
    "Who do you think will win the next election?",
    "Do you agree with the president's actions?",
    "What do you think about the opposition party?",
    "What's the weather like?",
    "How's your day going?",
    "It's a bit chilly today.",
    "Let's grab a coffee.",
    "What is the capital of France?",
    "how do I learn a language",
    "what is the best programming language?",
    "how long should I boil an egg?",
]
y = ["politics"] * 4 + ["chitchat"] * 4 + [None] * 4

router = IndexedRouter(encoder=encoder, routes=routes, confidence_threshold=0.5)
router.fit(X=X, y=y)
router.to_yaml(path="router.yaml")

The calibrator is now in the file:

In [12]:
with open("router.yaml") as f:
    print(f.read().split("routes:")[0])

router_type: indexed
encoder:
  type: fastembed
  name: BAAI/bge-small-en-v1.5
sparse_encoder: null
alpha: 0.3
top_k: 5
aggregation: mean
index:
  type: local
confidence_threshold: 0.5
calibrator:
  method: platt
  params:
    a:
    - 14.435170771055589
    b:
    - -7.982230249144389



And the loaded router reports calibrated confidences:

In [13]:
loaded = load_router(path="router.yaml")

router("how's the weather today?"), loaded("how's the weather today?")

(RouteChoice(name='chitchat', confidence=0.9743308193136435, calibrated=True),
 RouteChoice(name='chitchat', confidence=0.9743308193136435, calibrated=True))

The file holds the calibrator's fitted parameters, but not the queries it was fitted on. So if you `add()` or `remove()` routes on a loaded router, its calibrator is dropped with a warning: call `fit()` again.

---